In [20]:
import os
import json
import pandas as pd
import traceback

In [21]:
from langchain.chat_models import ChatOpenAI

In [22]:

from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2

In [23]:
KEY="***************************"     #Hidden to local env

In [24]:
llm= ChatOpenAI(openai_api_key=KEY,model_name= "gpt-3.5-turbo",temperature=0.3)

In [25]:
RESPONSE_JSON = {
    "1": {
        'name': 'name of the Medicine',
        'Content Value': 'Dosage value in mg (Milli Gram) or ml (Millilitre)',
        'Dosage': 'Dosage frequency'
    },
    "2": {
        'name': 'name of the Medicine',
        'Content Value': 'Dosage value in mg (Milli Gram) or ml (Millilitre)',
        'Dosage': 'Dosage frequency'
    },
    "3": {
        'name': 'name of the Medicine',
        'Content Value': 'Dosage value in mg (Milli Gram) or ml (Millilitre)',
        'Dosage': 'Dosage frequency (How many times in a day it should take)'
    },
}

In [26]:
TEXT="""
Name Armando
Cogna
Address West Rimbo Makati City
Age 29 Sex M Date 12-03-90
Px
Hinox
Amoxicillin Joong Cap 21
Sigj 1 cap 3x a day for
Sween days
Physicians Sig
delay
Lic No 123457
PTR No 127458467
2 No
"""

### This is start of chain

In [27]:
TEMPLATE="""
Text:{text}
You are an expert of Medicines. Given the above text, it is your job to extract the medicine information from it.
Make sure you extract all medicine from text, check all medicines Dosage value in mg and Dosage frequency carefully.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to extract all medicine from text.
### RESPONSE_JSON
{response_json}

"""

**Creating Langchain Chaining**

In [28]:
Prescription_prompt = PromptTemplate(
    input_variables=["text","response_json"],
    template=TEMPLATE
    )

In [29]:
precription_chain=LLMChain(llm=llm, prompt=Prescription_prompt, output_key="medicine", verbose=True)

In [30]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a name of the Medicines.\
You need to correct any spelling errors and provide the corrected medicines name.
Medicine Information:
{medicine}

Check from an expert English Writer of the above medicine information:
"""

In [31]:
precrition_evaluation_prompt=PromptTemplate(input_variables=["medicine"], template=TEMPLATE2)

In [32]:
review_chain=LLMChain(llm=llm, prompt=precrition_evaluation_prompt, output_key="review", verbose=True)

In [33]:
generate_evaluate_chain=SequentialChain(chains=[precription_chain, review_chain], input_variables=["text", "response_json"],
                                        output_variables=["medicine", "review"], verbose=True,)

In [34]:
#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain(
        {
            "text": TEXT,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text:
Name Armando
Cogna
Address West Rimbo Makati City
Age 29 Sex M Date 12-03-90
Px
Hinox
Amoxicillin Joong Cap 21
Sigj 1 cap 3x a day for
Sween days
Physicians Sig
delay
Lic No 123457
PTR No 127458467
2 No

You are an expert of Medicines. Given the above text, it is your job to extract the medicine information from it.
Make sure you extract all medicine from text, check all medicines Dosage value in mg and Dosage frequency carefully.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. Ensure to extract all medicine from text.
### RESPONSE_JSON
{"1": {"name": "name of the Medicine", "Content Value": "Dosage value in mg (Milli Gram) or ml (Millilitre)", "Dosage": "Dosage frequency"}, "2": {"name": "name of the Medicine", "Content Value": "Dosage value in mg (Milli Gram) or ml (Millilitre)", "Dosage": "Dosage frequency"}, "3": {"name": "name of the Medi

In [35]:
response_json = response.get("medicine")

In [36]:
response_json

'{\n    "1": {\n        "name": "Hinox",\n        "Content Value": "Cap 21",\n        "Dosage": "1 cap 3x a day for seven days"\n    },\n    "2": {\n        "name": "Amoxicillin Joong",\n        "Content Value": "Cap 21",\n        "Dosage": "1 cap 3x a day for seven days"\n    }\n}'

In [37]:
json_string= response_json

In [38]:
print(json_string)

{
    "1": {
        "name": "Hinox",
        "Content Value": "Cap 21",
        "Dosage": "1 cap 3x a day for seven days"
    },
    "2": {
        "name": "Amoxicillin Joong",
        "Content Value": "Cap 21",
        "Dosage": "1 cap 3x a day for seven days"
    }
}


# Using NLP lets check Drugs name efficiently

In [ ]:
from drug_named_entity_recognition import find_drugs
import spacy

nlp = spacy.blank("en")

# Your JSON data
# json_data = '''
# {
#   "1": {"name": "Betaloc", "Content Value": "50mg", "Dosage": "1 tab BID"},
#   "2": {"name": "Dorzolamidum", "Content Value": "10mg", "Dosage": "1 tab BID"},
#   "3": {"name": "Cimetidine", "Content Value": "50mg", "Dosage": "2 tabs TID"},
#   "4": {"name": "Oxprelol", "Content Value": "50mg", "Dosage": "1 tab QD"}
# }
# '''

# Extracting drug names from the JSON data
import json
data = json.loads(json_string)
drug_names = [data[key]["name"] for key in data]

# Check for drug names using the find_drugs function
output = find_drugs(drug_names, is_ignore_case=True)

# Print the result
print(output)


In [ ]:
from drug_named_entity_recognition import find_drugs
import spacy

nlp = spacy.blank("en")

# Your JSON data
json_data = '''
'\n### RESPONSE_JSON\n{"1": {"name": "CALPOL", "Content Value": "250 mg", "Dosage": "4 mL Q6H x 3 d"}, "2": {"name": "DELCON", "Content Value": "3 mL", "Dosage": "TDS x 5d"}, "3": {"name": "LEVOLIN", "Content Value": "3 mL", "Dosage": "TDS x 5d"}, "4": {"name": "MEFTAL-P", "Content Value": "100 mg", "Dosage": "3 mL SOS"}}'

'''

# Extracting drug names from the JSON data
import json
data = json.loads(json_data)
drug_names = [data[key]["name"] for key in data]

# Check for drug names using the find_drugs function
output = find_drugs(drug_names, is_ignore_case=True)

# Print the result
print(output)
